# Transform Yield Data

Reads raw maize yield CSVs for each location and reshapes them from wide to long form.
Columns like `A-N0 Zrnje` are split into `management` (A/B/C), `fertilization` (N0–N3), and `product` (Zrnje/Slama).
Outputs are saved to `data/interim/yield/`.

In [1]:
import pandas as pd

from aquacrop_slovenia import config
from aquacrop_slovenia.plots import plot_observed_yield

Loaded data sizes: Temperatures=1096, ETo=1096, Rain=1096


In [8]:
locations = {
    "jablje": config.RAW_YIELD_DIR / "maize-jablje.csv",
    "rakican": config.RAW_YIELD_DIR / "maize-rakican.csv",
}

product_translations = {
    "zrnje": "grain",
    "slama": "straw",
}

for location, path in locations.items():
    wide = pd.read_csv(path, sep=";", dtype={"Leto": int})

    long = wide.melt(id_vars="Leto", var_name="column", value_name="yield")

    # Column format: "{management}-{fertilization} {product}"
    # e.g. "A-N0 Zrnje" -> management=A, fertilization=N0, product=Zrnje
    split = long["column"].str.extract(r"^(?P<management>[A-C])-(?P<fertilization>N\d) (?P<product>.+)$")
    long = pd.concat([long.drop(columns="column"), split], axis=1)
    long = long.assign(product=long["product"].str.strip().str.lower().map(product_translations))

    long["location"] = location
    long = long.rename(columns={"Leto": "year"})
    long = long[["location", "year", "management", "fertilization", "product", "yield"]]
    long = long.dropna(subset=["yield"]).reset_index(drop=True)

    long["yield"] = long["yield"].astype(float) / 1000

    group_cols = ["location", "year", "management", "fertilization"]
    pivoted = (
        long[long["product"].isin(["grain", "straw"])]
        .pivot_table(index=group_cols, columns="product", values="yield")
        .reset_index()
    )
    pivoted.loc[:, "biomass_yield"] = pivoted["grain"] + pivoted["straw"]
    pivoted.loc[:, "hi_yield"] = pivoted["grain"] / pivoted["biomass_yield"] * 100

    biomass = pivoted[group_cols + ["biomass_yield"]].rename(columns={"biomass_yield": "yield"}).assign(product="biomass")
    hi = pivoted[group_cols + ["hi_yield"]].rename(columns={"hi_yield": "yield"}).assign(product="hi")

    long = pd.concat([long, biomass, hi], ignore_index=True)

    out_path = config.INTERIM_YIELD_DIR / f"maize-{location}.csv"
    long.to_csv(out_path, index=False)
    print(f"Saved {len(long)} rows to {out_path}")

Saved 1240 rows to C:\Users\rokku\Documents\Code\aquacrop-slovenia\data\interim\yield\maize-jablje.csv
Saved 1140 rows to C:\Users\rokku\Documents\Code\aquacrop-slovenia\data\interim\yield\maize-rakican.csv


C:\Users\rokku\AppData\Local\Temp\ipykernel_10792\2014716829.py:27: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  long["yield"] = long["yield"].astype(float) / 1000
C:\Users\rokku\AppData\Local\Temp\ipykernel_10792\2014716829.py:27: FutureWa

In [ ]:
long

In [4]:
dfs = [pd.read_csv(p) for p in sorted(config.INTERIM_YIELD_DIR.glob("maize-*.csv"))]
yield_df = pd.concat(dfs, ignore_index=True)

out_dir = config.PROCESSED_DIR / "yield"
out_path = out_dir / "maize.pkl"
yield_df.to_pickle(out_path)
print(f"Saved {len(yield_df)} rows to {out_path}")

Saved 2380 rows to C:\Users\rokuk\Documents\Code\aquacrop-slovenia\data\processed\yield\maize.pkl


In [4]:
#plot_observed_yield(yield_df, "grain")

In [5]:
#plot_observed_yield(yield_df, "biomass")